# Snowflake Dynamic Tables Tutorial

This notebook demonstrates **Dynamic Tables** — declarative, automatically-refreshing tables that simplify ELT pipelines.

**What you'll learn:**
1. Create Dynamic Tables with `TARGET_LAG` for automatic refresh
2. Build a star schema pipeline (fact + dimension tables)
3. Chain Dynamic Tables with `DOWNSTREAM` lag
4. Monitor refresh history and manage lifecycle

**Pattern:** Setup → Demo → Cleanup

## 1. Setup — Create Database, Schema, Warehouse & Source Data

In [ ]:
%%sql -r setup_result
CREATE OR REPLACE DATABASE dynamic_tables_tutorial_db;
CREATE SCHEMA dynamic_tables_tutorial_db.raw;
CREATE SCHEMA dynamic_tables_tutorial_db.staging;
CREATE SCHEMA dynamic_tables_tutorial_db.marts;
CREATE OR REPLACE WAREHOUSE dynamic_tables_wh
    WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;

In [ ]:
%%sql -r tables_result
CREATE OR REPLACE TABLE dynamic_tables_tutorial_db.raw.orders (
    order_id INT,
    customer_id INT,
    product_id INT,
    quantity INT,
    unit_price DECIMAL(10,2),
    order_date DATE,
    status VARCHAR(20)
);

CREATE OR REPLACE TABLE dynamic_tables_tutorial_db.raw.customers (
    customer_id INT,
    customer_name VARCHAR(100),
    segment VARCHAR(50),
    region VARCHAR(50)
);

CREATE OR REPLACE TABLE dynamic_tables_tutorial_db.raw.products (
    product_id INT,
    product_name VARCHAR(100),
    category VARCHAR(50),
    cost_price DECIMAL(10,2)
);

## 2. Insert Sample Data

In [ ]:
%%sql -r data_insert
INSERT INTO dynamic_tables_tutorial_db.raw.customers VALUES
    (1, 'Acme Corp', 'Enterprise', 'EMEA'),
    (2, 'Beta Ltd', 'Mid-Market', 'EMEA'),
    (3, 'Gamma Inc', 'SMB', 'APAC'),
    (4, 'Delta PLC', 'Enterprise', 'AMER'),
    (5, 'Epsilon SA', 'Mid-Market', 'APAC');

INSERT INTO dynamic_tables_tutorial_db.raw.products VALUES
    (101, 'Widget Pro', 'Hardware', 25.00),
    (102, 'Data Suite', 'Software', 150.00),
    (103, 'Cloud Pack', 'Services', 500.00),
    (104, 'Bolt Basic', 'Hardware', 10.00);

INSERT INTO dynamic_tables_tutorial_db.raw.orders VALUES
    (1001, 1, 101, 50, 45.00, '2026-01-15', 'COMPLETED'),
    (1002, 2, 102, 5, 200.00, '2026-01-20', 'COMPLETED'),
    (1003, 3, 103, 2, 750.00, '2026-02-01', 'COMPLETED'),
    (1004, 1, 104, 200, 15.00, '2026-02-10', 'SHIPPED'),
    (1005, 4, 101, 100, 42.00, '2026-02-15', 'COMPLETED'),
    (1006, 5, 102, 10, 190.00, '2026-03-01', 'PENDING'),
    (1007, 2, 103, 1, 700.00, '2026-03-10', 'COMPLETED'),
    (1008, 3, 104, 500, 12.00, '2026-03-15', 'SHIPPED');

## 3. Create Dynamic Tables — Staging Layer

Dynamic Tables declare the transformation SQL and a `TARGET_LAG` that controls how fresh the data should be.

In [ ]:
%%sql -r stg_orders
CREATE OR REPLACE DYNAMIC TABLE dynamic_tables_tutorial_db.staging.stg_orders
    WAREHOUSE = dynamic_tables_wh
    TARGET_LAG = '1 minute'
AS
SELECT
    order_id,
    customer_id,
    product_id,
    quantity,
    unit_price,
    quantity * unit_price AS line_total,
    order_date,
    status
FROM dynamic_tables_tutorial_db.raw.orders;

In [ ]:
%%sql -r stg_customers
CREATE OR REPLACE DYNAMIC TABLE dynamic_tables_tutorial_db.staging.stg_customers
    WAREHOUSE = dynamic_tables_wh
    TARGET_LAG = '1 minute'
AS
SELECT
    customer_id,
    customer_name,
    segment,
    region
FROM dynamic_tables_tutorial_db.raw.customers;

In [ ]:
%%sql -r stg_orders_data
SELECT * FROM dynamic_tables_tutorial_db.staging.stg_orders;

## 4. Create Dynamic Tables — Marts Layer (Star Schema)

The mart tables use `DOWNSTREAM` lag, meaning they refresh automatically after their upstream dependencies refresh.

In [ ]:
%%sql -r dim_customers
CREATE OR REPLACE DYNAMIC TABLE dynamic_tables_tutorial_db.marts.dim_customers
    WAREHOUSE = dynamic_tables_wh
    TARGET_LAG = DOWNSTREAM
AS
SELECT
    c.customer_id,
    c.customer_name,
    c.segment,
    c.region,
    COUNT(o.order_id) AS total_orders,
    SUM(o.line_total) AS lifetime_revenue
FROM dynamic_tables_tutorial_db.staging.stg_customers c
LEFT JOIN dynamic_tables_tutorial_db.staging.stg_orders o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name, c.segment, c.region;

In [ ]:
%%sql -r fct_sales
CREATE OR REPLACE DYNAMIC TABLE dynamic_tables_tutorial_db.marts.fct_daily_sales
    WAREHOUSE = dynamic_tables_wh
    TARGET_LAG = '1 minute'
AS
SELECT
    o.order_date,
    p.category AS product_category,
    c.region AS customer_region,
    COUNT(o.order_id) AS order_count,
    SUM(o.line_total) AS total_revenue,
    AVG(o.line_total) AS avg_order_value
FROM dynamic_tables_tutorial_db.staging.stg_orders o
JOIN dynamic_tables_tutorial_db.raw.products p ON o.product_id = p.product_id
JOIN dynamic_tables_tutorial_db.staging.stg_customers c ON o.customer_id = c.customer_id
GROUP BY o.order_date, p.category, c.region;

In [ ]:
%%sql -r dim_data
SELECT * FROM dynamic_tables_tutorial_db.marts.dim_customers
ORDER BY lifetime_revenue DESC;

In [ ]:
%%sql -r fct_data
SELECT * FROM dynamic_tables_tutorial_db.marts.fct_daily_sales
ORDER BY order_date;

## 5. Test Automatic Refresh — Insert New Data

In [ ]:
%%sql -r new_orders
INSERT INTO dynamic_tables_tutorial_db.raw.orders VALUES
    (1009, 1, 103, 3, 725.00, '2026-03-20', 'PENDING'),
    (1010, 4, 102, 20, 185.00, '2026-03-22', 'COMPLETED');

In [ ]:
%%sql -r refreshed_data
SELECT * FROM dynamic_tables_tutorial_db.marts.fct_daily_sales
ORDER BY order_date DESC
LIMIT 5;

## 6. Monitor Dynamic Tables

In [ ]:
%%sql -r show_dt
SHOW DYNAMIC TABLES IN DATABASE dynamic_tables_tutorial_db;

In [ ]:
%%sql -r refresh_history
SELECT *
FROM TABLE(dynamic_tables_tutorial_db.information_schema.dynamic_table_refresh_history())
ORDER BY refresh_start_time DESC
LIMIT 10;

## 7. Manage Dynamic Tables — Suspend, Resume, Manual Refresh

In [ ]:
%%sql -r manage_dt
ALTER DYNAMIC TABLE dynamic_tables_tutorial_db.staging.stg_orders SUSPEND;
ALTER DYNAMIC TABLE dynamic_tables_tutorial_db.staging.stg_orders RESUME;
ALTER DYNAMIC TABLE dynamic_tables_tutorial_db.staging.stg_orders REFRESH;

## 8. Cleanup — Drop All Tutorial Objects

In [ ]:
%%sql -r cleanup_result
DROP DATABASE IF EXISTS dynamic_tables_tutorial_db;
DROP WAREHOUSE IF EXISTS dynamic_tables_wh;